# European Steel Decarbonization
## Temporal Action Score Analysis: Pre-COVID vs Post-COVID

**Team:** SuSteelAible  
**Author:** Irene  
**Date:** December 2024  

---

## Overview

This notebook applies the Action Score framework to two distinct periods:
- **Pre-COVID (2013-2019):** Baseline decarbonization dynamics
- **Post-COVID (2020-2024):** Acceleration period with policy changes

By comparing scores across these periods, we can identify:
- Which companies accelerated decarbonization efforts
- How transformation announcements translated to action
- Whether the policy landscape shift (EU Green Deal, CBAM) impacted trajectories

The analysis culminates in an **animated visualization** showing company movement in the Talk vs Action matrix from 2019 → 2024.

---

## Scoring Methodology

Same framework as main notebook (100 points total):
1. **Performance (30 pts):** Final year intensity for each period (2019 vs 2024)
2. **Trend (30 pts):** Improvement rate within each period
3. **Data Quality (15 pts):** Reporting completeness for each period
4. **Technology (20 pts):** Status as of end of period (2019 vs 2024)
5. **Renewable (0-5 pts):** Renewable % for each period (2019 vs 2024)

**Key difference:** Trend scores calculated separately for each period to capture acceleration/deceleration.

---

Let's begin.

## 1. Setup & Data Loading

In [11]:
# Setup
import sys
from pathlib import Path

# Add scripts folder to path
scripts_path = Path.cwd().parent / 'scripts'
sys.path.insert(0, str(scripts_path))

In [12]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import seaborn as sns
import subprocess

# Import custom modules
from data_loader import (
    load_company_data,
    filter_complete_data,
    prepare_analysis_dataset,
    print_data_summary
)

# Import scoring functions
from action_score_functions import (
    calculate_performance_score,
    calculate_trend_score,
    calculate_data_quality_score,
    calculate_renewable_score
)

In [13]:
# Load and prepare data using data_loader
df_raw = load_company_data(fix_apostrophes=True, filter_region='Europe')
df_filtered = filter_complete_data(df_raw, min_years=4)
df_analysis = prepare_analysis_dataset(df_filtered)
print_data_summary(df_analysis)

# Settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

# Define periods
PRE_PERIOD = (2013, 2019)
POST_PERIOD = (2020, 2024)

companies = sorted(df_analysis['company'].unique())

print(f"\n📊 PERIOD DEFINITIONS:")
print(f"  Pre-COVID: {PRE_PERIOD[0]}-{PRE_PERIOD[1]}")
print(f"  Post-COVID: {POST_PERIOD[0]}-{POST_PERIOD[1]}")
print(f"\n  Companies: {len(companies)}")

   ℹ️  Filtered out 4 rows with missing Scope 1 data
--------------------------------------------------------------------------------
DATASET SUMMARY
--------------------------------------------------------------------------------

🏭 Companies: 13
📂 Total rows: 103
📅 Year range: 2013-2024

Technologies: BF-BOF, EAF, EAF Stainless
Countries: Austria, Bulgaria-Greece, Finland, Germany, Italy, Luxembourg, Netherlands, Spain, Sweden, UK
--------------------------------------------------------------------------------

📊 PERIOD DEFINITIONS:
  Pre-COVID: 2013-2019
  Post-COVID: 2020-2024

  Companies: 13


c:\Users\irene\Documents\neue_fische\susteelaible\scripts\data_loader.py:295: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['scope2_method_used'] = df.groupby('company').apply(


## 2. Load Technology Scores

In [14]:
# Load technology scores
tech_scores = pd.read_excel(
    '../data/emissions_and_production_technology.xlsx',
    sheet_name='technology_scores',
    engine='openpyxl'
)

# Filter to European companies
tech_scores = tech_scores[tech_scores['company'].isin(companies)].copy()

print(f"✓ Technology scores loaded for {len(tech_scores)} companies")
print(f"  Columns: {list(tech_scores.columns)}")

✓ Technology scores loaded for 13 companies
  Columns: ['company', 'country', 'current_tech', 'transformation_plan', 'plan_status', 'tech_score_2019', 'tech_score_2024', 'renewable_pct_2019', 'renewable_pct_2024', 'notes_2019', 'notes_2024']


## 3. Calculate Scores for Both Periods

We'll calculate all 5 components for both pre-COVID (2013-2019) and post-COVID (2020-2024) periods.

In [15]:
def calculate_period_scores(df_raw, df_analysis, tech_scores, companies, period_name,
                          start_year, end_year, renewable_col, use_long_term_trend=True):
    """
    Calculate all 5 action score components for a specific period.

    IMPORTANT: Only scores companies with sufficient data IN THIS PERIOD.
    Companies without enough years in a period will get NaN scores to avoid
    misleading comparisons.

    Parameters
    ----------
    df_raw : pd.DataFrame
        Raw data (for data quality assessment)
    df_analysis : pd.DataFrame
        Prepared analysis dataset
    tech_scores : pd.DataFrame
        Technology scores
    companies : list
        List of company names
    period_name : str
        Period identifier ('pre2020' or 'post2020')
    start_year : int
        Period start year
    end_year : int
        Period end year (used for performance score)
    renewable_col : str
        Column name for renewable % ('renewable_pct_2019' or 'renewable_pct_2024')
    use_long_term_trend : bool, optional
        If True and period is post-2020, calculate trend from 2013 onwards
        to capture long-term commitment rather than just post-COVID fluctuations.
        Default: False

    Returns
    -------
    pd.DataFrame
        Complete action scores for all companies in this period
    """
    print(f"\n{'='*70}")
    print(f"CALCULATING {period_name.upper()} PERIOD SCORES ({start_year}-{end_year})")
    print(f"{'='*70}")

    results = []

    for company in companies:
        # ====================================================================
        # Check if company has sufficient data in each period
        # ====================================================================
        company_period_data = df_analysis[
            (df_analysis['company'] == company) &
            (df_analysis['year'] >= start_year) &
            (df_analysis['year'] <= end_year)
        ]

        n_years_in_period = len(company_period_data)

        # Skip company if insufficient data in this period
        if n_years_in_period < 3:
            print(f"  ⚠️  {company}: Only {n_years_in_period} years in {start_year}-{end_year} - SKIPPING")
            results.append({
                'company': company,
                'period': period_name,
                'performance_score': np.nan,
                'intensity': np.nan,
                'trend_score': np.nan,
                'slope_pct': np.nan,
                'r_squared': np.nan,
                'data_quality_score': np.nan,
                'n_years': n_years_in_period,
                'technology_score': np.nan,
                'renewable_score': np.nan,
                'renewable_pct': np.nan,
                'total_score': np.nan,
                'skip_reason': f'Only {n_years_in_period} years in period (need ≥3)'
            })
            continue

        # ====================================================================
        # Company has sufficient data - calculate scores
        # ====================================================================

        # Component 1: Performance (final year of period)
        perf = calculate_performance_score(df_analysis, company, year=end_year)

        # Component 2: Trend (within period OR long-term if specified)
        # For post-COVID, optionally use 2013-2024 to capture long-term commitment
        if use_long_term_trend and start_year >= 2020:
            trend_start = 2013
            print(f"    → {company}: Using long-term trend (2013-{end_year})")
        else:
            trend_start = start_year

        trend = calculate_trend_score(df_analysis, company, start_year=trend_start, end_year=end_year)

        # Component 3: Data Quality (use same time window as trend for consistency)
        if use_long_term_trend and start_year >= 2020:
            quality_start = 2013
        else:
            quality_start = start_year

        quality = calculate_data_quality_score(df_raw, company, start_year=quality_start, end_year=end_year)

        # Component 4: Technology (load from Excel - period-specific column)
        tech_row = tech_scores[tech_scores['company'] == company]
        if len(tech_row) > 0:
            # Use period-specific technology score if available
            # Column names: tech_score_2019, tech_score_2024
            tech_col = f'tech_score_{end_year}' if f'tech_score_{end_year}' in tech_scores.columns else 'tech_score'
            if tech_col in tech_row.columns:
                tech_score = tech_row[tech_col].values[0]
            else:
                tech_score = 0
        else:
            tech_score = 0

        # Component 5: Renewable (period-specific)
        renewable = calculate_renewable_score(company, tech_scores, renewable_col=renewable_col)

        # Combine results
        total_score = (
            perf['performance_score'] +
            trend['trend_score'] +
            quality['data_quality_score'] +
            tech_score +
            renewable['renewable_score']
        )

        results.append({
            'company': company,
            'period': period_name,
            'performance_score': perf['performance_score'],
            'intensity': perf['intensity'],
            'trend_score': trend['trend_score'],
            'slope_pct': trend.get('slope_pct', np.nan),
            'r_squared': trend.get('r_squared', np.nan),
            'data_quality_score': quality['data_quality_score'],
            'n_years': n_years_in_period,
            'technology_score': tech_score,
            'renewable_score': renewable['renewable_score'],
            'renewable_pct': renewable.get('renewable_pct', np.nan),
            'total_score': total_score
        })

        print(f"  ✓ {company}: {n_years_in_period} years, Total = {total_score:.1f} pts")

    df_period = pd.DataFrame(results)

    # Summary statistics (excluding skipped companies)
    valid_scores = df_period[df_period['total_score'].notna()]

    print(f"\n{'='*70}")
    print(f"✓ Calculated scores for {len(valid_scores)} companies")
    print(f"⚠️  Skipped {len(df_period) - len(valid_scores)} companies (insufficient data)")
    if len(valid_scores) > 0:
        print(f"  Mean total score: {valid_scores['total_score'].mean():.1f} / 100")
        print(f"  Score range: {valid_scores['total_score'].min():.1f} - {valid_scores['total_score'].max():.1f}")
    print(f"{'='*70}")

    return df_period

In [16]:
# Calculate pre-COVID scores (2013-2019)
pre_scores = calculate_period_scores(
    df_raw, df_analysis, tech_scores, companies,
    period_name='pre2020',
    start_year=2013,
    end_year=2019,
    renewable_col='renewable_pct_2019'
)


CALCULATING PRE2020 PERIOD SCORES (2013-2019)
  ⚠️  Acciaierie d'Italia Holding: Only 1 years in 2013-2019 - SKIPPING
  ⚠️  Acerinox EU: Only 1 years in 2013-2019 - SKIPPING
  ✓ ArcelorMittal: 7 years, Total = 28.4 pts
  ⚠️  Celsa Group: Only 0 years in 2013-2019 - SKIPPING
  ✓ Feralpi Group: 7 years, Total = 69.0 pts
  ✓ Outokumpu: 7 years, Total = 73.6 pts
  ✓ SHS Group: 3 years, Total = 14.6 pts
  ⚠️  SIDENOR Group: Only 0 years in 2013-2019 - SKIPPING
  ✓ SSAB: 7 years, Total = 28.9 pts
  ✓ Salzgitter AG: 4 years, Total = 19.7 pts
  ⚠️  Tata Steel Nederland: Only 1 years in 2013-2019 - SKIPPING
  ⚠️  Tata Steel UK: Only 0 years in 2013-2019 - SKIPPING
  ✓ Voestalpine: 3 years, Total = 14.1 pts

✓ Calculated scores for 7 companies
⚠️  Skipped 6 companies (insufficient data)
  Mean total score: 35.5 / 100
  Score range: 14.1 - 73.6


In [17]:
# Calculate post-COVID scores (2020-2024)
post_scores = calculate_period_scores(
    df_raw, df_analysis, tech_scores, companies,
    period_name='post2020',
    start_year=2020,
    end_year=2024,
    renewable_col='renewable_pct_2024'
)


CALCULATING POST2020 PERIOD SCORES (2020-2024)
    → Acciaierie d'Italia Holding: Using long-term trend (2013-2024)
  ✓ Acciaierie d'Italia Holding: 3 years, Total = 19.0 pts
    → Acerinox EU: Using long-term trend (2013-2024)
  ✓ Acerinox EU: 5 years, Total = 75.3 pts
    → ArcelorMittal: Using long-term trend (2013-2024)
  ✓ ArcelorMittal: 5 years, Total = 32.0 pts
    → Celsa Group: Using long-term trend (2013-2024)
  ✓ Celsa Group: 5 years, Total = 50.7 pts
    → Feralpi Group: Using long-term trend (2013-2024)
  ✓ Feralpi Group: 5 years, Total = 69.1 pts
    → Outokumpu: Using long-term trend (2013-2024)
  ✓ Outokumpu: 5 years, Total = 92.5 pts
    → SHS Group: Using long-term trend (2013-2024)
  ✓ SHS Group: 5 years, Total = 28.6 pts
    → SIDENOR Group: Using long-term trend (2013-2024)
  ✓ SIDENOR Group: 5 years, Total = 79.0 pts
    → SSAB: Using long-term trend (2013-2024)
  ✓ SSAB: 5 years, Total = 40.1 pts
    → Salzgitter AG: Using long-term trend (2013-2024)
  ✓ Salzgit

## 4. Period Comparison

Compare how companies' scores changed between the two periods.

In [18]:
# Combine both periods
action_scores_temporal = pd.concat([pre_scores, post_scores], ignore_index=True)

# Calculate changes for companies with both periods
comparison = []

for company in companies:
    pre = pre_scores[pre_scores['company'] == company]
    post = post_scores[post_scores['company'] == company]

    if len(pre) > 0 and len(post) > 0:
        comparison.append({
            'company': company,
            'pre_total': pre['total_score'].values[0],
            'post_total': post['total_score'].values[0],
            'change': post['total_score'].values[0] - pre['total_score'].values[0],
            'pre_trend': pre['trend_score'].values[0],
            'post_trend': post['trend_score'].values[0],
            'trend_change': post['trend_score'].values[0] - pre['trend_score'].values[0]
        })

comparison_df = pd.DataFrame(comparison).sort_values('change', ascending=False)

print("\n" + "="*70)
print("PERIOD COMPARISON: WHO IMPROVED?")
print("="*70)
print(comparison_df[['company', 'pre_total', 'post_total', 'change']].to_string(index=False))
print("="*70)


PERIOD COMPARISON: WHO IMPROVED?
                    company  pre_total  post_total    change
                  Outokumpu  73.587396   92.500000 18.912604
              Salzgitter AG  19.699000   34.550815 14.851815
                  SHS Group  14.552556   28.608696 14.056140
                       SSAB  28.900643   40.065939 11.165295
                Voestalpine  14.123288   22.000000  7.876712
              ArcelorMittal  28.369710   31.975881  3.606171
              Feralpi Group  69.047244   69.082859  0.035615
Acciaierie d'Italia Holding        NaN   19.000000       NaN
                Acerinox EU        NaN   75.301370       NaN
                Celsa Group        NaN   50.705967       NaN
              SIDENOR Group        NaN   79.022260       NaN
       Tata Steel Nederland        NaN   24.812785       NaN
              Tata Steel UK        NaN   12.404682       NaN


## 5. Save Combined Scores

Export the complete temporal dataset for the animation.

In [19]:
# Save combined scores
action_scores_temporal.to_csv('../results/models/action_scores_temporal.csv', index=False)
print("✓ Saved: ../results/models/action_scores_temporal.csv")

# Save comparison
comparison_df.to_csv('../results/models/action_scores_comparison.csv', index=False)
print("✓ Saved: ../results/models/action_scores_comparison.csv")

✓ Saved: ../results/models/action_scores_temporal.csv
✓ Saved: ../results/models/action_scores_comparison.csv


## 6. Animated Visualization: Talk vs Action (2019 → 2024)

The final visualization shows company movement in the Talk vs Action space from 2019 to 2024.

**Data requirements:**
- Action scores: from this notebook (just calculated)
- Talk scores: from ClimateBERT analysis (separate analysis)

**Note:** If ClimateBERT data is not available, this section will be skipped.

In [20]:
# Load ClimateBERT output (year-wise data)
climateBERT_output = pd.read_csv('../results/bert/company_year.csv')

# Aggregate into periods
def assign_period(year):
    if year <= 2019:
        return 'pre2020'
    else:
        return 'post2020'

climateBERT_output['period'] = climateBERT_output['year'].apply(assign_period)

# Aggregate by company and period (mean climate percentage)
climate_agg_df = climateBERT_output.groupby(['company', 'period']).agg({
    'climate_pct': 'mean'  # Or whatever your column is called
}).reset_index()

# Rename column to match what function expects
climate_agg_df = climate_agg_df.rename(columns={'climate_pct': 'climate_pct_mean'})

print("\n✓ ClimateBERT data aggregated:")
print(climate_agg_df.head())
print(f"\nCompanies: {climate_agg_df['company'].nunique()}")
print(f"Periods: {climate_agg_df['period'].unique()}")


✓ ClimateBERT data aggregated:
             company    period  climate_pct_mean
0  AcciaieriedItalia  post2020         72.943783
1           Acerinox  post2020         48.484544
2           Acerinox   pre2020         18.060674
3      ArcelorMittal  post2020         62.797941
4      ArcelorMittal   pre2020         48.746961

Companies: 15
Periods: ['post2020' 'pre2020']


In [21]:
from plotting_utils import (
    prepare_animation_data,
    create_animated_talk_action_matrix,
    ANIMATION_TECH_MAP
)

# Prepare data
action_ready, talk_ready = prepare_animation_data(
    action_scores_temporal,
    climate_agg_df
)

# Create animation with custom settings
output_path = create_animated_talk_action_matrix(
    action_scores_df=action_ready,
    talk_scores_df=talk_ready,
    technology_map=ANIMATION_TECH_MAP,
    output_path='../results/models/talk_vs_action.mp4',
    total_frames=180,  # Shorter = faster
    fps=20,  # Smoother
    pause_start=30,  # Longer pause at start
    pause_end=80  # Longer pause at end
)

# Display in notebook
from IPython.display import Video
display(Video(output_path, width=800, embed=True))

✓ Animation data prepared with SuSteelAible project mappings
✓ ffmpeg found

✓ Animation data prepared:
  Companies in 2019: 8
  Companies in 2024: 13
  New reporters: 5
    Acciaierie, Celsa, SIDENOR, Tata NL, Tata UK

Generating animation (180 frames at 20 FPS)...
Saving as MP4 (this may take ~1 minute)...

✅ Saved: ../results/models/talk_vs_action.mp4

Video details:
  Duration: 9.0 seconds
  Resolution: 1920x1440 (16:12 aspect ratio)
  Format: MP4 (H.264)


## Talk vs Action Matrix - Interpretation

### Key Patterns (2019 → 2024)

#### **The Technology Divide**
- **EAF companies (green)** cluster in the **right half** (action >50), demonstrating that cleaner technology enables higher operational performance.
- **BF-BOF companies (orange)** remain **concentrated in the left half** (action <50), reflecting structural constraints of carbon-intensive technology.
- This spatial separation visualizes **technology lock-in**: traditional steelmakers face barriers to high action scores regardless of climate commitments.

#### **Universal Talk Increase**
- Nearly all companies show **upward movement** (~10-20 percentage points more climate discussion).
- The increase in "talk" is **technology-neutral**, reflecting mainstreaming of climate discourse across the sector.
- Driven by stakeholder pressure, regulatory requirements (EU Green Deal, CBAM), and evolving disclosure standards.

#### **Divergent Trajectories**
- **Some BF-BOF companies**: Primarily vertical movement (more talk, limited action improvement).
- **Some EAF companies**: Diagonal movement (both more talk AND more action).
- A few companies: Horizontal movement (improved action with stable talk) - "quiet decarbonization."

### Implications

This visualization suggests that **external pressures drive talk uniformly**, but **internal factors (technology, capital) determine action**. The persistent technology divide confirms that **technology transformation, not incremental efficiency**, is required to move from left to right on the action dimension.